# 학습 루프의 정석: Fashion-MNIST

**30일 딥러닝 완전 정복 · Day 16** — 강의 본문: [https://muy-bien-hwa.github.io/For-my-Claude/](https://muy-bien-hwa.github.io/For-my-Claude/)

### 이 노트북에서 할 일
- 재사용 가능한 학습 템플릿(train_one_epoch / evaluate / fit)을 만든다
- 시드 3개로 반복해 평균 ± 표준편차를 보고한다
- 체크포인트를 Google Drive에 저장하고 이어서 학습한다
- TensorBoard로 학습 과정을 기록한다

> ⚙️ **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 선택하세요. 셀은 위에서부터 `Shift+Enter`로 실행합니다.
> 그래프 라벨은 Colab 기본 글꼴 문제를 피하려고 영어로 표기했습니다.

## 1. 템플릿
이 셀은 이후 노트북들(Day 18, 19, 21, 29)에서도 그대로 다시 씁니다. 각 줄의 의미를 강의 본문과 대조해 보세요.

In [ ]:
import random, os, json, time
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from tqdm.auto import tqdm
device = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def train_one_epoch(model, loader, loss_fn, opt, scheduler=None, clip=None, scaler=None):
    model.train(); tot, correct, n = 0.0, 0, 0
    for xb, yb in tqdm(loader, leave=False):
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=scaler is not None):
            logits = model(xb); loss = loss_fn(logits, yb)
        opt.zero_grad(set_to_none=True)
        if scaler is not None:
            scaler.scale(loss).backward()
            if clip: scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(opt); scaler.update()
        else:
            loss.backward()
            if clip: torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
        if scheduler is not None: scheduler.step()
        tot += loss.item() * len(yb); correct += (logits.argmax(1) == yb).sum().item(); n += len(yb)
    return tot / n, correct / n

@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval(); tot, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb); tot += loss_fn(logits, yb).item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item(); n += len(yb)
    return tot / n, correct / n

def save_ckpt(path, model, opt, epoch, best):
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": epoch, "best": best}, path)

def load_ckpt(path, model, opt=None):
    ck = torch.load(path, map_location=device)
    model.load_state_dict(ck["model"])
    if opt is not None: opt.load_state_dict(ck["opt"])
    return ck["epoch"], ck["best"]

def fit(model, train_dl, val_dl, epochs, lr=1e-3, wd=0.05, ckpt="best.pt", patience=None, label_smoothing=0.0, amp=True, sgd=False):
    loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    if sgd: opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True, weight_decay=wd)
    else:   opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_dl), pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda") if (amp and device == "cuda") else None
    hist = {k: [] for k in ["tr_loss", "tr_acc", "va_loss", "va_acc"]}; best, bad = 0.0, 0
    for ep in range(epochs):
        t0 = time.time()
        tl, ta = train_one_epoch(model, train_dl, loss_fn, opt, sched, clip=1.0, scaler=scaler)
        vl, va = evaluate(model, val_dl, nn.CrossEntropyLoss())
        for k, v in zip(hist, [tl, ta, vl, va]): hist[k].append(v)
        print(f"[{ep+1:02d}/{epochs}] train {tl:.4f}/{ta:.3f}  val {vl:.4f}/{va:.3f}  lr {opt.param_groups[0]['lr']:.2e}  {time.time()-t0:.0f}s")
        if va > best: best, bad = va, 0; save_ckpt(ckpt, model, opt, ep, best)
        else:
            bad += 1
            if patience and bad >= patience: print("early stopping"); break
    load_ckpt(ckpt, model)
    return hist

def plot_hist(hist, title=""):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].plot(hist["tr_loss"], label="train"); ax[0].plot(hist["va_loss"], label="val"); ax[0].set_title(f"{title} loss"); ax[0].legend()
    ax[1].plot(hist["tr_acc"], label="train"); ax[1].plot(hist["va_acc"], label="val"); ax[1].set_title(f"{title} accuracy"); ax[1].legend()
    plt.show()
print("template ready on", device)

## 2. 데이터: Fashion-MNIST
옷·신발·가방 등 10종류의 28×28 흑백 이미지. MNIST보다 어렵습니다.

In [ ]:
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2860,), (0.3530,))])
full = datasets.FashionMNIST("data", train=True, download=True, transform=tf)
test_ds = datasets.FashionMNIST("data", train=False, download=True, transform=tf)
train_ds, val_ds = random_split(full, [55000, 5000], generator=torch.Generator().manual_seed(42))
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_dl = DataLoader(val_ds, batch_size=512, num_workers=2)
test_dl = DataLoader(test_ds, batch_size=512, num_workers=2)
classes = full.classes
xb, yb = next(iter(train_dl))
print(xb.shape, yb.shape, f"min {xb.min():.2f} max {xb.max():.2f}", classes)

import matplotlib.pyplot as plt
fig, axs = plt.subplots(2, 8, figsize=(14, 4))
for a, x, y in zip(axs.ravel(), xb, yb): a.imshow(x[0], cmap="gray"); a.set_title(classes[y], fontsize=8); a.axis("off")
plt.show()

## 3. 모델과 학습

In [ ]:
def make_mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 512), nn.ReLU(), nn.Dropout(0.3),
                         nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 10)).to(device)
seed_everything(0)
model = make_mlp()
hist = fit(model, train_dl, val_dl, epochs=15, lr=2e-3, wd=0.05, ckpt="fmnist_best.pt")
plot_hist(hist, "Fashion-MNIST MLP")
print("test (loss, acc):", evaluate(model, test_dl, nn.CrossEntropyLoss()))

## 4. 시드 3개로 신뢰할 수 있는 결과 내기

In [ ]:
accs = []
for seed in [0, 1, 2]:
    seed_everything(seed); m = make_mlp()
    h = fit(m, train_dl, val_dl, epochs=10, lr=2e-3, ckpt=f"seed{seed}.pt")
    accs.append(max(h["va_acc"]))
print(f"val acc = {np.mean(accs):.4f} ± {np.std(accs):.4f}  (seeds: {np.round(accs, 4)})")

## 5. Google Drive에 체크포인트 저장 → 이어서 학습
Colab 세션이 끊기면 로컬 파일은 사라집니다. 중요한 체크포인트는 Drive에 두세요.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/dl30", exist_ok=True)
path = "/content/drive/MyDrive/dl30/fmnist.pt"
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
save_ckpt(path, model, opt, epoch=15, best=max(hist["va_acc"]))
print("saved to", path)

# (런타임을 다시 시작했다고 가정) 새 모델을 만들고 불러와서 이어서 학습
model2 = make_mlp(); opt2 = torch.optim.AdamW(model2.parameters(), lr=1e-4)
ep, best = load_ckpt(path, model2, opt2)
print("resumed from epoch", ep, "best", best, "→ val", evaluate(model2, val_dl, nn.CrossEntropyLoss()))

## 6. TensorBoard

In [ ]:
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter("runs/fmnist")
for i, (tl, vl, va) in enumerate(zip(hist["tr_loss"], hist["va_loss"], hist["va_acc"])):
    writer.add_scalar("loss/train", tl, i); writer.add_scalar("loss/val", vl, i); writer.add_scalar("acc/val", va, i)
writer.close()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 7. 과제
- 검증 정확도 89% 이상을 달성했나요? 못 했다면 무엇을 바꿔 볼지 Day 28 진단표를 참고하세요.
- `fit`의 `patience=3`을 주고 조기 종료가 언제 일어나는지 보세요.
- 이 템플릿 셀을 복사해 두세요. Day 29 캡스톤에서 그대로 씁니다.